In [1]:
# import the required libraries
import os

# need to be installed
import yaml
import joblib
import numpy as np
import pandas as pd

from tqdm import tqdm
from sklearn.model_selection import train_test_split

## **1 -  Configuration File**
---

In [2]:
def load_config(config_path):
    """
    Load the configuration file.

    Parameters :
    ----------
    config_path : str (string)
        configuration file location

    Return:
    -------
    Params : dict
        Loaded configuration file.
        
    """
    # try to load config.yaml file
    try:
        with open(config_path, 'r') as file:
            params = yaml.safe_load(file)
    except FileNotFoundError as err:
        raise RuntimeError(f"Configuration File Not Found in {config_path}")

    return params

In [55]:
def update_config(key, value, params, config_path):
    # to maintan the raw  config  file immutable (berubah ketika dipanggil saja)
    params = params.copy()

    # Update the Configuration
    params[key] = value

    # Write the Configuration File
    with open(config_path, 'w') as file:
        yaml.dump(params, file)

    print(f"Params Updates! Key:{key} - Value: {value}")

    # Reload the Updated Configuration file.
    config = load_config(config_path)

    return config

In [56]:
#load the config.yaml
PATH_CONFIG = "../config/config.yaml"
config = load_config(PATH_CONFIG)

In [57]:
config["path_raw_data"]

'../data/raw/'

In [58]:
config

{'columns_datetime': ['tanggal'],
 'features': ['stasiun', 'pm10', 'pm25', 'so2', 'co', 'o3', 'no2'],
 'columns_int': ['pm10', 'pm25', 'so2', 'co', 'o3', 'no2', 'max'],
 'label': 'category',
 'label_categories': ['BAIK', 'SEDANG', 'TIDAK SEHAT'],
 'label_categories_new': ['BAIK', 'TIDAK BAIK'],
 'object_columns': ['stasiun', 'critical', 'category'],
 'path_joined_data': '../data/interim/joined_dataset.pkl',
 'path_raw_data': '../data/raw/',
 'path_validated_data': '../data/interim/validated_data.pkl',
 'range_co': [0, 100],
 'range_no2': [0, 100],
 'range_o3': [0, 140],
 'range_pm10': [0, 800],
 'range_pm25': [0, 400],
 'range_so2': [0, 500],
 'range_stasiun': ['DKI1 (Bunderan HI)',
  'DKI2 (Kelapa Gading)',
  'DKI3 (Jagakarsa)',
  'DKI4 (Lubang Buaya)',
  'DKI5 (Kebon Jeruk) Jakarta Barat']}

## **2 -  Data Collection**
---

In [7]:
def load_data(data_path):
    # Create data frame. 
    raw_dataset = pd.DataFrame()

    for i in tqdm(os.listdir(data_path)):
        raw_dataset = pd.concat([
            pd.read_csv(data_path + i), raw_dataset
        ])

    return raw_dataset
        

In [8]:
PATH_RAW_DATA = config["path_raw_data"]
raw_dataset = load_data(data_path = PATH_RAW_DATA)

100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 12/12 [00:00<00:00, 150.17it/s]


In [9]:
raw_dataset

,tanggal,stasiun,pm10,pm25,so2,co,o3,no2,max,critical,categori
0,2021-11-01,DKI1 (Bunderan HI),58,85,32,16,22,41,85,PM25,SEDANG
1,2021-11-02,DKI1 (Bunderan HI),50,63,29,14,22,35,63,PM25,SEDANG
2,2021-11-03,DKI1 (Bunderan HI),49,63,33,13,21,38,63,PM25,SEDANG
3,2021-11-04,DKI1 (Bunderan HI),57,83,30,17,34,47,83,PM25,SEDANG
4,2021-11-05,DKI1 (Bunderan HI),56,75,31,19,18,46,75,PM25,SEDANG
...,...,...,...,...,...,...,...,...,...,...,...
150,2021-07-27,DKI5 (Kebon Jeruk) Jakarta Barat,64,110,32,13,29,35,110,PM25,TIDAK SEHAT
151,2021-07-28,DKI5 (Kebon Jeruk) Jakarta Barat,70,130,33,17,28,45,130,PM25,TIDAK SEHAT
152,2021-07-29,DKI5 (Kebon Jeruk) Jakarta Barat,78,140,32,18,29,39,140,PM25,TIDAK SEHAT
153,2021-07-30,DKI5 (Kebon Jeruk) Jakarta Barat,75,121,37,12,50,21,121,PM25,TIDAK SEHAT


In [10]:
raw_dataset = raw_dataset.reset_index(drop=True)
raw_dataset

,tanggal,stasiun,pm10,pm25,so2,co,o3,no2,max,critical,categori
0,2021-11-01,DKI1 (Bunderan HI),58,85,32,16,22,41,85,PM25,SEDANG
1,2021-11-02,DKI1 (Bunderan HI),50,63,29,14,22,35,63,PM25,SEDANG
2,2021-11-03,DKI1 (Bunderan HI),49,63,33,13,21,38,63,PM25,SEDANG
3,2021-11-04,DKI1 (Bunderan HI),57,83,30,17,34,47,83,PM25,SEDANG
4,2021-11-05,DKI1 (Bunderan HI),56,75,31,19,18,46,75,PM25,SEDANG
...,...,...,...,...,...,...,...,...,...,...,...
1825,2021-07-27,DKI5 (Kebon Jeruk) Jakarta Barat,64,110,32,13,29,35,110,PM25,TIDAK SEHAT
1826,2021-07-28,DKI5 (Kebon Jeruk) Jakarta Barat,70,130,33,17,28,45,130,PM25,TIDAK SEHAT
1827,2021-07-29,DKI5 (Kebon Jeruk) Jakarta Barat,78,140,32,18,29,39,140,PM25,TIDAK SEHAT
1828,2021-07-30,DKI5 (Kebon Jeruk) Jakarta Barat,75,121,37,12,50,21,121,PM25,TIDAK SEHAT


In [11]:
PATH_JOINED_DATA = "../data/interim/joined_dataset.pkl"
joblib.dump(raw_dataset, PATH_JOINED_DATA)

['../data/interim/joined_dataset.pkl']

In [12]:
# Update the Configuration parameter
config = update_config(
    key = "path_joined_data",
    value = PATH_JOINED_DATA,
    params = config,
    config_path = PATH_CONFIG
    
)

Params Updates! Key:path_joined_data - Value: ../data/interim/joined_dataset.pkl


## **3 -  Data Validation**
---

In [13]:
# Check the data type for each future
raw_dataset.dtypes

tanggal        str
stasiun        str
pm10        object
pm25        object
so2         object
co          object
o3          object
no2         object
max         object
critical       str
categori       str
dtype: object

## **3.1 -  Handling Column Tanggal**
---

In [14]:
# Try to cast the column to datetime type. / mengubah tanggal menjadi datetime agar dapat diolah 
raw_dataset["tanggal"] = pd.to_datetime(raw_dataset["tanggal"])

## **3.2 -  Handling Column pm10**
---

In [15]:
# Try to cast the column to int type.
raw_dataset["pm10"] = raw_dataset["pm10"].astype(int)

ValueError: invalid literal for int() with base 10: '---'

In [16]:
# Ensure no single data that is -1. 
# didalam kolom pm10 masih terdapat data "---" atau data kosong atau rusak dll
raw_dataset.eq("-1").any() | raw_dataset.eq(-1).any()  # code ini untuk mengecek apakah ada nilai -1 ? jika false artinya tidak ada data -1

tanggal     False
stasiun     False
pm10        False
pm25        False
so2         False
co          False
o3          False
no2         False
max         False
critical    False
categori    False
dtype: bool

In [17]:
# Replace the "---" with -1 and cast the column into int.
# code ini memastikan "---" diganti menjadi -1 dan seluruh kolom diubah menjadi int
raw_dataset["pm10"] = raw_dataset["pm10"].replace("---", -1).astype(int)

In [ ]:
raw_dataset["pm10"].dtype # cara memastikan code berjalan

In [18]:
print("Tipe data:", raw_dataset["pm10"].dtype)
print("Masih ada '---'? ", (raw_dataset["pm10"] == "---").any())
print("Jumlah -1:", (raw_dataset["pm10"] == -1).sum())


Tipe data: int64
Masih ada '---'?  False
Jumlah -1: 70


## **3.3 -  Handling Column pm25**
---

In [19]:
# Replace the "---" with -1 and cast the column into int.
raw_dataset["pm25"] = raw_dataset["pm25"].replace("---", -1).astype(int)

ValueError: cannot convert float NaN to integer

In [20]:
# Sanity check the missing values. / cara cek berapa jumlah NaN
raw_dataset["pm25"].isna().sum()

np.int64(62)

In [21]:
# Replace the NaN values with -1.
raw_dataset["pm25"] = raw_dataset["pm25"].fillna(-1)

# Sanity check the missing values.
raw_dataset["pm25"].isna().sum()

np.int64(0)

In [22]:
raw_dataset["pm25"] = raw_dataset["pm25"].replace("---", -1).astype(int)

In [23]:
raw_dataset["pm25"].unique()

array([ 85,  63,  83,  75,  72,  56,  55,  73,  49,  70,  53,  39,  47,
        71,  40,  44,  54,  33,  30,  35,  57,  37,  20,  28,  50,  88,
        -1,  95,  84,  62,  86,  59,  58,  78,  80,  81,  45,  68,  74,
        61,  52,  65,  82,  66,  76,  41,  25,  31,  77,  93, 106, 100,
        99,  38,  91,  69,  87,  34,  43,  67,  36,  26,  60,  46,  64,
        96,  94,  90, 116, 124, 136, 102,  98, 121,  89,  42,  79, 110,
       108, 112,  97,  23,  24,  21, 113, 117, 105,  92, 103, 107, 118,
       115, 109, 101, 104,  27, 114, 119, 134, 123, 122, 140, 111, 126,
        51, 138,  19,  22, 125,  48,  29,  16,  13, 129, 120, 132, 141,
       161, 153, 130, 145, 156, 174, 150, 147, 148, 131, 154, 127, 157])

In [24]:
raw_dataset["pm25"].dtype

dtype('int64')

In [25]:
print("Jumlah -1:", (raw_dataset["pm25"] == -1).sum())

Jumlah -1: 103


## **3.4 -  Handling Column so2**
---

In [26]:
# Try to cast the column to int type.
raw_dataset["so2"] = raw_dataset["so2"].astype(int)

ValueError: invalid literal for int() with base 10: '---'

In [27]:
# Replace the "---" with -1 and cast the column into int.
# code ini memastikan "---" diganti menjadi -1 dan seluruh kolom diubah menjadi int
raw_dataset["so2"] = raw_dataset["so2"].replace("---", -1).astype(int)

In [28]:
raw_dataset["so2"].dtype # cara memastikan code berjalan

dtype('int64')

## **3.5 -  Handling Column co**
---

In [29]:
# Try to cast the column to int type.
raw_dataset["co"] = raw_dataset["co"].astype(int)

ValueError: invalid literal for int() with base 10: '---'

In [30]:
raw_dataset["co"] = raw_dataset["co"].replace("---", -1).astype(int)

In [31]:
raw_dataset["co"].dtype # cara memastikan code berjalan

dtype('int64')

## **3.6 -  Handling Column o3**
---

In [32]:
# Try to cast the column to int type.
raw_dataset["o3"] = raw_dataset["o3"].astype(int)

ValueError: invalid literal for int() with base 10: '---'

In [33]:
raw_dataset["o3"] = raw_dataset["o3"].replace("---", -1).astype(int)

In [34]:
raw_dataset["o3"].dtype # cara memastikan code berjalan

dtype('int64')

## **3.7 -  Handling Column no2**
---

In [35]:
# Try to cast the column to int type.
raw_dataset["no2"] = raw_dataset["no2"].astype(int)

ValueError: invalid literal for int() with base 10: '---'

In [36]:
raw_dataset["no2"] = raw_dataset["no2"].replace("---", -1).astype(int)

In [37]:
raw_dataset["o3"].dtype # cara memastikan code berjalan

dtype('int64')

## **3.8 -  Handling Column max**
---

In [38]:
# Try to cast the column to int type.
raw_dataset["max"] = raw_dataset["max"].astype(int)

ValueError: invalid literal for int() with base 10: 'PM25'

In [39]:
# Check which data that cause error.
raw_dataset[raw_dataset["max"] == "PM25"]

,tanggal,stasiun,pm10,pm25,so2,co,o3,no2,max,critical,categori
152,2021-12-03,DKI1 (Bunderan HI),49,31,9,19,7,49,PM25,BAIK,NaN


In [40]:
raw_dataset.sample(5)

,tanggal,stasiun,pm10,pm25,so2,co,o3,no2,max,critical,categori
165,2021-12-16,DKI1 (Bunderan HI),56,74,39,20,24,16,74,PM25,SEDANG
1104,2021-07-30,DKI1 (Bunderan HI),73,105,29,12,39,35,105,PM25,TIDAK SEHAT
448,2021-05-20,DKI5 (Kebon Jeruk) Jakarta Barat,51,87,26,8,28,16,87,PM25,SEDANG
38,2021-11-09,DKI2 (Kelapa Gading),61,86,-1,11,47,28,86,PM25,SEDANG
221,2021-12-10,DKI3 (Jagakarsa),51,65,47,12,-1,15,65,PM25,SEDANG


In [ ]:
# dari 2 tabel diatas ada typo terbalik isian kolomnya
# kolom critical harusnya di isi PM25 kolom categori di isi BAIK

In [41]:
raw_dataset["max"].unique() # cara ngecek isian yg unik selain angka

array([85, 63, 83, 75, 72, 56, 55, 73, 49, 70, 53, 39, 47, 71, 40, 44, 54,
       33, 32, 35, 57, 37, 29, 30, 50, 88, 52, 95, 84, 62, 86, 59, 58, 78,
       80, 81, 45, 68, 42, 74, 61, 65, 82, 66, 41, 76, 46, 0, 77, 93, 106,
       100, 99, 38, 91, 69, 87, 34, 43, 67, 36, 26, '65', '34', 'PM25',
       '52', '57', '81', '60', '50', '59', '43', '46', '61', '75', '74',
       '71', '63', '85', '77', '53', '68', '54', '55', '88', '58', '51',
       '62', '86', '0', '64', '96', '67', '94', '99', '56', '70', '66',
       '37', '91', '44', '80', '90', '72', '82', '78', '116', '100', '83',
       '179', '76', '73', '124', '136', '69', '102', '98', '121', '89',
       '42', '45', '84', '87', 79, 110, 108, 60, 64, 112, 90, 97, 116,
       102, 151, 89, 96, 94, 113, 117, 105, 92, 103, 107, 118, 115, 109,
       101, 98, 104, 114, 119, 134, 123, 122, 140, 111, 126, 51, 138, 125,
       48, 20, 31, 25, 17, 27, 28, 129, 120, 124, 132, 121, 141, 161, 153,
       130, 145, 156, 174, 150, 147, 148, 13

In [42]:
# Fix the error.
raw_dataset.loc[152, "max"] = raw_dataset.loc[152, "pm10"]
raw_dataset.loc[152, "critical"] = "PM10"
raw_dataset.loc[152, "categori"] = "BAIK"

In [43]:
# Sanity check the result.
raw_dataset.loc[152]

tanggal     2021-12-03 00:00:00
stasiun      DKI1 (Bunderan HI)
pm10                         49
pm25                         31
so2                           9
co                           19
o3                            7
no2                          49
max                          49
critical                   PM10
categori                   BAIK
Name: 152, dtype: object

In [44]:
# Cast the column to int.
raw_dataset["max"] = raw_dataset["max"].astype(int)

## **3.9 -  Handling Column critical**
---

In [45]:
# Check the unique value.
raw_dataset["critical"].value_counts()

critical
PM25    1631
PM10      65
O3        57
CO        34
SO2       26
Name: count, dtype: int64

## **3.10 -  Handling Column categori**
---

In [46]:
# Check the unique value.
raw_dataset["categori"].value_counts()

categori
SEDANG            1305
TIDAK SEHAT        319
BAIK               189
TIDAK ADA DATA      17
Name: count, dtype: int64

In [47]:
# Drop the "TIDAK ADA DATA" category.
missing_labels = raw_dataset[raw_dataset["categori"] == "TIDAK ADA DATA"]
raw_dataset = raw_dataset.drop(index = missing_labels.index)

In [48]:
# Sanity check the result.
raw_dataset["categori"].value_counts()

categori
SEDANG         1305
TIDAK SEHAT     319
BAIK            189
Name: count, dtype: int64

In [50]:
raw_dataset.info()

<class 'pandas.DataFrame'>
Index: 1813 entries, 0 to 1829
Data columns (total 11 columns):
 #   Column    Non-Null Count  Dtype         
---  ------    --------------  -----         
 0   tanggal   1813 non-null   datetime64[us]
 1   stasiun   1813 non-null   str           
 2   pm10      1813 non-null   int64         
 3   pm25      1813 non-null   int64         
 4   so2       1813 non-null   int64         
 5   co        1813 non-null   int64         
 6   o3        1813 non-null   int64         
 7   no2       1813 non-null   int64         
 8   max       1813 non-null   int64         
 9   critical  1813 non-null   str           
 10  category  1813 non-null   str           
dtypes: datetime64[us](1), int64(7), str(3)
memory usage: 170.0 KB


In [49]:
# Rename "categori" into "category".
raw_dataset = raw_dataset.rename(columns = {"categori": "category"})
raw_dataset.info()

<class 'pandas.DataFrame'>
Index: 1813 entries, 0 to 1829
Data columns (total 11 columns):
 #   Column    Non-Null Count  Dtype         
---  ------    --------------  -----         
 0   tanggal   1813 non-null   datetime64[us]
 1   stasiun   1813 non-null   str           
 2   pm10      1813 non-null   int64         
 3   pm25      1813 non-null   int64         
 4   so2       1813 non-null   int64         
 5   co        1813 non-null   int64         
 6   o3        1813 non-null   int64         
 7   no2       1813 non-null   int64         
 8   max       1813 non-null   int64         
 9   critical  1813 non-null   str           
 10  category  1813 non-null   str           
dtypes: datetime64[us](1), int64(7), str(3)
memory usage: 170.0 KB


In [51]:
# Update the configuration parameter.
config = update_config(
    key = "label",
    value = "category",
    params = config,
    config_path = PATH_CONFIG
)

Params Updates! Key:label - Value: category


In [59]:
# Update the configuration parameter.
col_object = config["object_columns"]
col_object[-1] = "category"

config = update_config(
    key = "object_columns",
    value = col_object,
    params = config,
    config_path = PATH_CONFIG
)

Params Updates! Key:object_columns - Value: ['stasiun', 'critical', 'category']


In [60]:
# Sanity check the data types.
raw_dataset.info()

<class 'pandas.DataFrame'>
Index: 1813 entries, 0 to 1829
Data columns (total 11 columns):
 #   Column    Non-Null Count  Dtype         
---  ------    --------------  -----         
 0   tanggal   1813 non-null   datetime64[us]
 1   stasiun   1813 non-null   str           
 2   pm10      1813 non-null   int64         
 3   pm25      1813 non-null   int64         
 4   so2       1813 non-null   int64         
 5   co        1813 non-null   int64         
 6   o3        1813 non-null   int64         
 7   no2       1813 non-null   int64         
 8   max       1813 non-null   int64         
 9   critical  1813 non-null   str           
 10  category  1813 non-null   str           
dtypes: datetime64[us](1), int64(7), str(3)
memory usage: 170.0 KB


In [61]:
# Serialized the validated data.
PATH_VALIDATED_DATA = f"../data/interim/validated_data.pkl"
joblib.dump(raw_dataset, PATH_VALIDATED_DATA)

['../data/interim/validated_data.pkl']

In [62]:
# Update the configuration parameter.
config = update_config(
    key = "path_validated_data",
    value = PATH_VALIDATED_DATA,
    params = config,
    config_path = PATH_CONFIG
)

Params Updates! Key:path_validated_data - Value: ../data/interim/validated_data.pkl


## **4 - Update the Range of Data in Configuration File**
---

In [63]:
# Update the range of data with the min and max value of each column.
cols = ["pm10", "pm25", "so2", "co", "o3", "no2"]
param_keys = ["range_pm10", "range_pm25", "range_so2", "range_co", "range_o3", "range_no2"]

for col, key in zip(cols, param_keys):
    config = update_config(
        key = key,
        value = [int(np.min(raw_dataset[col])), int(np.max(raw_dataset[col]))],
        params = config,
        config_path = PATH_CONFIG
    )

Params Updates! Key:range_pm10 - Value: [-1, 179]
Params Updates! Key:range_pm25 - Value: [-1, 174]
Params Updates! Key:range_so2 - Value: [-1, 82]
Params Updates! Key:range_co - Value: [-1, 47]
Params Updates! Key:range_o3 - Value: [-1, 151]
Params Updates! Key:range_no2 - Value: [-1, 65]


In [64]:
# Check the configuration parameters.
config

{'columns_datetime': ['tanggal'],
 'columns_int': ['pm10', 'pm25', 'so2', 'co', 'o3', 'no2', 'max'],
 'features': ['stasiun', 'pm10', 'pm25', 'so2', 'co', 'o3', 'no2'],
 'label': 'category',
 'label_categories': ['BAIK', 'SEDANG', 'TIDAK SEHAT'],
 'label_categories_new': ['BAIK', 'TIDAK BAIK'],
 'object_columns': ['stasiun', 'critical', 'category'],
 'path_joined_data': '../data/interim/joined_dataset.pkl',
 'path_raw_data': '../data/raw/',
 'path_validated_data': '../data/interim/validated_data.pkl',
 'range_co': [-1, 47],
 'range_no2': [-1, 65],
 'range_o3': [-1, 151],
 'range_pm10': [-1, 179],
 'range_pm25': [-1, 174],
 'range_so2': [-1, 82],
 'range_stasiun': ['DKI1 (Bunderan HI)',
  'DKI2 (Kelapa Gading)',
  'DKI3 (Jagakarsa)',
  'DKI4 (Lubang Buaya)',
  'DKI5 (Kebon Jeruk) Jakarta Barat']}

## **5 - Data Defense**
---

In [ ]:
# Create the check_data() function.
# It receives 2 arguments: input_data and params
#     - input_data is the raw dataset
#     - params is the configuration parameters
# It is a void function (no return value).
#If AssertionError happens, there are exists data that don't match the configuration

In [67]:
# Function to do data defense.
def check_data(input_data, params):
    """
    Do data defense for checking the data types and range of data.

    Parameters:
    ----------
    input_data : pd.DataFrame
        The data to be checked.

    params : dict
        Loaded configuration parameters.

    Returns:
    -------
    None, it's a void function.
    """

    # Check data types.
    assert input_data.select_dtypes("datetime").columns.to_list() == params["columns_datetime"], "an error occurs in datetime column(s)."
    assert input_data.select_dtypes("str").columns.to_list() == params["object_columns"], "an error occurs in object column(s)."
    assert input_data.select_dtypes("number").columns.to_list() == params["columns_int"], "an error occurs in int32 column(s)."

    # Check range of data.
    assert set(input_data['stasiun']).issubset(set(params['range_stasiun'])), "an error occurs in stasiun range."
    assert input_data['pm10'].between(params['range_pm10'][0], params['range_pm10'][1]).sum() == len(input_data), "an error occurs in pm10 range."
    assert input_data['pm25'].between(params['range_pm25'][0], params['range_pm25'][1]).sum() == len(input_data), "an error occurs in pm25 range."
    assert input_data['so2'].between(params['range_so2'][0], params['range_so2'][1]).sum() == len(input_data), "an error occurs in so2 range."
    assert input_data['co'].between(params['range_co'][0], params['range_co'][1]).sum() == len(input_data), "an error occurs in co range."
    assert input_data['o3'].between(params['range_o3'][0], params['range_o3'][1]).sum() == len(input_data), "an error occurs in o3 range."
    assert input_data['no2'].between(params['range_no2'][0], params['range_no2'][1]).sum() == len(input_data), "an error occurs in no2 range."

In [68]:
# Do data defense.
check_data(raw_dataset, config)

In [70]:
# Input-Output Split.
def split_input_output(input_data, params):
    """
    Split the input(X) and output (y).

    Parameters:
    ----------
    input_data : pd.DataFrame
        The processed dataset.

    params : dict
        Loaded configuration parameters.

    Returns:
    -------
    X : pd.DataFrame
        The input data.

    y : pd.Series
        The output data.
    """

    X = input_data[params["features"]].copy()
    y = input_data[params["label"]].copy()

    print(f"Original data shape : {input_data.shape}")
    print(f"Selected Features   : {params['features']}")
    print(f"X data shape        : {X.shape}")
    print(f"y data shape        : {y.shape}")

    return X, y

In [71]:
X, y = split_input_output(
    input_data = raw_dataset,
    params = config
)

Original data shape : (1813, 11)
Selected Features   : ['stasiun', 'pm10', 'pm25', 'so2', 'co', 'o3', 'no2']
X data shape        : (1813, 7)
y data shape        : (1813,)


In [72]:
# Sanity check the input (X).
X.head()

,stasiun,pm10,pm25,so2,co,o3,no2
0,DKI1 (Bunderan HI),58,85,32,16,22,41
1,DKI1 (Bunderan HI),50,63,29,14,22,35
2,DKI1 (Bunderan HI),49,63,33,13,21,38
3,DKI1 (Bunderan HI),57,83,30,17,34,47
4,DKI1 (Bunderan HI),56,75,31,19,18,46


In [73]:
# Sanity check the output (y).
y.head()

0    SEDANG
1    SEDANG
2    SEDANG
3    SEDANG
4    SEDANG
Name: category, dtype: str

In [74]:
# Train-Test Split.
def split_train_test(X, y, test_size, random_state):
    """
    Split the train and test set.

    Parameters:
    ----------
    X : pd.DataFrame
        The input data.

    y : pd.Series
        The output data.

    test_size : float
        The proportion of test set.

    random_state : int
        For reproducibility

    Returns:
    -------
    X_train, X_test : pd.DataFrame
        The train and test input.

    y_train, y_test : pd.Series
        The train and test output.
    """

    X_train, X_test, y_train, y_test = train_test_split(
                                            X, y,
                                            test_size = test_size,
                                            random_state = random_state,
                                            stratify = y
                                       )

    print(f"X_train shape : {X_train.shape}")
    print(f"y_train shape : {y_train.shape}")
    print(f"X_test shape  : {X_test.shape}")
    print(f"y_test shape  : {y_test.shape}")

    return X_train, X_test, y_train, y_test


In [75]:
# train:valid:test -> 80:10:10

RANDOM_STATE = 123

# Train vs not-train.
X_train, X_not_train, y_train, y_not_train = split_train_test(
    X = X,
    y = y,
    test_size = 0.2,
    random_state = RANDOM_STATE
)

print()
# Valid vs test.
X_valid, X_test, y_valid, y_test = split_train_test(
    X = X_not_train,
    y = y_not_train,
    test_size = 0.5,
    random_state = RANDOM_STATE
)

X_train shape : (1450, 7)
y_train shape : (1450,)
X_test shape  : (363, 7)
y_test shape  : (363,)

X_train shape : (181, 7)
y_train shape : (181,)
X_test shape  : (182, 7)
y_test shape  : (182,)


In [76]:
# Serialize the splitted data.
PATH_SPLITTED_DATA = f"../data/interim/"

joblib.dump(X_train, f"{PATH_SPLITTED_DATA}X_train.pkl")
joblib.dump(y_train, f"{PATH_SPLITTED_DATA}y_train.pkl")
joblib.dump(X_valid, f"{PATH_SPLITTED_DATA}X_valid.pkl")
joblib.dump(y_valid, f"{PATH_SPLITTED_DATA}y_valid.pkl")
joblib.dump(X_test, f"{PATH_SPLITTED_DATA}X_test.pkl")
joblib.dump(y_test, f"{PATH_SPLITTED_DATA}y_test.pkl")

['../data/interim/y_test.pkl']

In [78]:
# Update the configuration parameters.
config = update_config(
    key = "path_train_set",
    value = [f"{PATH_SPLITTED_DATA}X_train.pkl", f"{PATH_SPLITTED_DATA}y_train.pkl"],
    params = config,
    config_path = PATH_CONFIG
)

config = update_config(
    key = "path_valid_set",
    value = [f"{PATH_SPLITTED_DATA}X_valid.pkl", f"{PATH_SPLITTED_DATA}y_valid.pkl"],
    params = config,
    config_path = PATH_CONFIG
)

config = update_config(
    key = "path_test_set",
    value = [f"{PATH_SPLITTED_DATA}X_test.pkl", f"{PATH_SPLITTED_DATA}y_test.pkl"],
    params = config,
    config_path = PATH_CONFIG
)

Params Updates! Key:path_train_set - Value: ['../data/interim/X_train.pkl', '../data/interim/y_train.pkl']
Params Updates! Key:path_valid_set - Value: ['../data/interim/X_valid.pkl', '../data/interim/y_valid.pkl']
Params Updates! Key:path_test_set - Value: ['../data/interim/X_test.pkl', '../data/interim/y_test.pkl']


In [79]:
# Check the configuration parameters.
config

{'columns_datetime': ['tanggal'],
 'columns_int': ['pm10', 'pm25', 'so2', 'co', 'o3', 'no2', 'max'],
 'features': ['stasiun', 'pm10', 'pm25', 'so2', 'co', 'o3', 'no2'],
 'label': 'category',
 'label_categories': ['BAIK', 'SEDANG', 'TIDAK SEHAT'],
 'label_categories_new': ['BAIK', 'TIDAK BAIK'],
 'object_columns': ['stasiun', 'critical', 'category'],
 'path_joined_data': '../data/interim/joined_dataset.pkl',
 'path_raw_data': '../data/raw/',
 'path_test_set': ['../data/interim/X_test.pkl', '../data/interim/y_test.pkl'],
 'path_train_set': ['../data/interim/X_train.pkl',
  '../data/interim/y_train.pkl'],
 'path_valid_set': ['../data/interim/X_valid.pkl',
  '../data/interim/y_valid.pkl'],
 'path_validated_data': '../data/interim/validated_data.pkl',
 'range_co': [-1, 47],
 'range_no2': [-1, 65],
 'range_o3': [-1, 151],
 'range_pm10': [-1, 179],
 'range_pm25': [-1, 174],
 'range_so2': [-1, 82],
 'range_stasiun': ['DKI1 (Bunderan HI)',
  'DKI2 (Kelapa Gading)',
  'DKI3 (Jagakarsa)',
  'DKI4